<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">2. Spark DataFrames and SQL</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 2.1 Lecture - Introduction to DataFrames
### Duration - 13 Minutes

> *Structured data processing in Apache Spark*

This lecture introduces the DataFrame, the primary data abstraction in modern Spark, and explains how DataFrame operations are lazily evaluated and optimized before they run.

## Learning Objectives

**By the end of this lecture, you will be able to:**

- Describe the DataFrame abstraction, including schema enforcement and distributed storage
- Distinguish transformations from actions and explain lazy evaluation
- Describe how the Catalyst optimizer and Photon engine optimize DataFrame execution
- Explain why columnar storage suits analytical workloads

## A. What is a DataFrame?

<p style="font-size: 1em; line-height: 1.6; color: #333">DataFrames are distributed collections of records, all with the same pre-defined structure (schema):</p>

- Built on Spark's core concepts but with structure, optimization, and familiar SQL-like operations for data manipulation
- DataFrames track their schema and provide native support for many common SQL functions and relational operators (`JOIN`, `GROUP BY`, and so on)
- DataFrames are evaluated as DAGs, using *lazy evaluation*, and carry lineage information that provides fault tolerance
- A DataFrame looks like a single table to you, but its rows are physically split into partitions spread across the Executors of the cluster

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">One logical table, many physical partitions</strong>
      </div>
    </div>
  </summary>
  <div style="margin: 8px 0 4px 0">
    <svg viewBox="0 0 720 290" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="One DataFrame split into three partitions across executors" style="width:100%; max-width:720px; height:auto; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif">
      <defs>
        <marker id="part-arrow" markerWidth="10" markerHeight="8" refX="7" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker>
      </defs>
      <line x1="360" y1="88" x2="120" y2="200" stroke="#618794" stroke-width="2" marker-end="url(#part-arrow)"/>
      <line x1="360" y1="88" x2="360" y2="200" stroke="#618794" stroke-width="2" marker-end="url(#part-arrow)"/>
      <line x1="360" y1="88" x2="600" y2="200" stroke="#618794" stroke-width="2" marker-end="url(#part-arrow)"/>
      <rect x="180" y="20" width="360" height="68" rx="8" fill="#DCE0E2" stroke="#618794" stroke-width="2"/>
      <text x="360" y="49" text-anchor="middle" font-size="17" font-weight="700" fill="#1B5162">DataFrame</text>
      <text x="360" y="71" text-anchor="middle" font-size="12.5" font-style="italic" fill="#555">logical view: named, typed columns</text>
      <rect x="20" y="205" width="200" height="64" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="120" y="234" text-anchor="middle" font-size="15" font-weight="600" fill="#333">Partition 0</text>
      <text x="120" y="255" text-anchor="middle" font-size="12.5" fill="#555">on Executor 1</text>
      <rect x="260" y="205" width="200" height="64" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="360" y="234" text-anchor="middle" font-size="15" font-weight="600" fill="#333">Partition 1</text>
      <text x="360" y="255" text-anchor="middle" font-size="12.5" fill="#555">on Executor 2</text>
      <rect x="500" y="205" width="200" height="64" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="600" y="234" text-anchor="middle" font-size="15" font-weight="600" fill="#333">Partition 2</text>
      <text x="600" y="255" text-anchor="middle" font-size="12.5" fill="#555">on Executor 3</text>
    </svg>
  </div>
</details>

## B. Creating DataFrames

<p style="font-size: 1em; line-height: 1.6; color: #333">DataFrames can be created from:</p>

- JSON, CSV, Parquet, ORC, text, or binary files
- Delta Lake or other table storage format directories
- A table or view in a catalog (like Unity Catalog)
- An external database
- An existing RDD (classic compute only)

<div class="code-block" data-language="python">
# From a file
df = spark.read.csv("/Volumes/my_catalog/my_schema/files/people.csv")
<br />
# From a table
df = spark.read.table("my_catalog.my_schema.people")
<br />
# From a range of numbers
df = spark.range(1000)
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-tomorrow.min.css" rel="stylesheet" />
<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" id="prism-light" disabled />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'python';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="theme-toggle" style="position:absolute;top:8px;left:8px;padding:4px 10px;font-size:12px;background:#333;color:#fff;border:1px solid #666;border-radius:4px;cursor:pointer;z-index:10;">◑</button>' +
                '<pre class="code-pre" data-theme="dark" style="background:#1e1e1e;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);

        var toggleBtn = block.querySelector('.theme-toggle');
        toggleBtn.onclick = function() {
            var pre = block.querySelector('.code-pre');
            var isDark = pre.getAttribute('data-theme') !== 'light';
            pre.setAttribute('data-theme', isDark ? 'light' : 'dark');
            pre.style.background = isDark ? '#f8f8f8' : '#1e1e1e';
            this.textContent = isDark ? '◐' : '◑';
            this.style.background = isDark ? '#ddd' : '#333';
            this.style.color = isDark ? '#333' : '#fff';
            document.getElementById('prism-light').disabled = !isDark;
            document.querySelector('link[href*="prism-tomorrow"]').disabled = isDark;
            Prism.highlightElement(codeEl);
        };
    });
})();
</script>

## C. Transformations and Actions

<p style="font-size: 1em; line-height: 1.6; color: #333">DataFrames are <strong>immutable</strong> - once created, their data cannot be modified. Instead:</p>

- **Transformations** create new DataFrames from existing ones; they describe *what* to compute, not *when*
- **Actions** (like counting, showing, or saving output) trigger actual computation and produce final results

<p style="font-size: 1em; line-height: 1.6; color: #333">Multiple transformations can be chained; the job is only created when an action is requested - a process called <strong>lazy evaluation</strong>.</p>

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Lazy evaluation: transformations build a plan, actions run it</strong>
      </div>
    </div>
  </summary>
  <div style="margin: 8px 0 4px 0; overflow-x: auto">
    <svg viewBox="0 0 1090 200" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Transformations build a plan lazily; an action triggers the Spark job" style="width:100%; max-width:1090px; height:auto; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif">
      <defs>
        <marker id="lazy-arrow" markerWidth="10" markerHeight="8" refX="7" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker>
      </defs>
      <rect x="10" y="40" width="680" height="128" rx="10" fill="#F9F7F4" stroke="#618794" stroke-width="2"/>
      <text x="350" y="64" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1B5162">Transformations — build the plan, nothing executes</text>
      <rect x="25" y="92" width="140" height="52" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="95" y="123" text-anchor="middle" font-size="14" font-family="Consolas,Monaco,monospace" fill="#1B5162">spark.read</text>
      <rect x="185" y="92" width="140" height="52" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="255" y="123" text-anchor="middle" font-size="14" font-family="Consolas,Monaco,monospace" fill="#1B5162">select()</text>
      <rect x="345" y="92" width="140" height="52" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="415" y="123" text-anchor="middle" font-size="14" font-family="Consolas,Monaco,monospace" fill="#1B5162">filter()</text>
      <rect x="505" y="92" width="140" height="52" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="575" y="123" text-anchor="middle" font-size="14" font-family="Consolas,Monaco,monospace" fill="#1B5162">withColumn()</text>
      <line x1="165" y1="118" x2="183" y2="118" stroke="#618794" stroke-width="2" marker-end="url(#lazy-arrow)"/>
      <line x1="325" y1="118" x2="343" y2="118" stroke="#618794" stroke-width="2" marker-end="url(#lazy-arrow)"/>
      <line x1="485" y1="118" x2="503" y2="118" stroke="#618794" stroke-width="2" marker-end="url(#lazy-arrow)"/>
      <line x1="648" y1="118" x2="738" y2="118" stroke="#618794" stroke-width="2" marker-end="url(#lazy-arrow)"/>
      <rect x="740" y="90" width="150" height="56" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="815" y="114" text-anchor="middle" font-size="15" font-family="Consolas,Monaco,monospace" fill="#333">count()</text>
      <text x="815" y="133" text-anchor="middle" font-size="12" font-style="italic" fill="#555">an Action</text>
      <line x1="890" y1="118" x2="918" y2="118" stroke="#618794" stroke-width="2" marker-end="url(#lazy-arrow)"/>
      <rect x="920" y="90" width="160" height="56" rx="8" fill="#FFCC66" stroke="#CD7F32" stroke-width="2"/>
      <text x="1000" y="114" text-anchor="middle" font-size="14" font-weight="700" fill="#333">Spark job runs</text>
      <text x="1000" y="133" text-anchor="middle" font-size="12.5" fill="#333">on the cluster</text>
    </svg>
  </div>
</details>
<br/>
<details>
<summary style="cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
</div>
</div>
</summary>
<div style="border-left: 4px solid #1B5162; padding: 4px 24px 14px 24px; margin: -16px 0 16px 0">
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 12px 0 4px 0">Why Lazy Evaluation Matters</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0">Because Spark sees the whole chain of transformations before anything runs, the optimizer can rearrange, combine, and prune work - for example pushing a filter down to the file scan so less data is read in the first place.</li>
<li style="margin: 4px 0">Lineage (the recorded chain of transformations) is also what makes DataFrames fault tolerant: a lost partition can be recomputed by replaying its lineage.</li>
<li style="margin: 4px 0">A common surprise for new users: defining ten transformations takes milliseconds and runs nothing. The cost is paid when an action such as <code>count()</code>, <code>display()</code>, or <code>write</code> executes the accumulated plan.</li>
</ul>
</div>
</details>

### Common DataFrame API Methods

The most frequently used transformations:

| Transformation | Description |
|----------------|-------------|
| __`select()`__ | Choose specific columns from a DataFrame |
| __`filter()`__ | Filter rows based on a condition, alias `where()` |
| __`withColumn()`__ | Add or modify a column in a DataFrame |
| __`groupBy()`__ | Group rows in a DataFrame by specified columns |
| __`agg()`__ | Aggregate rows in a DataFrame after grouping |

And the most frequently used actions, which trigger execution:

| Action | Description |
|--------|-------------|
| __`count()`__ | Return the number of rows in a DataFrame |
| __`show()`__ | Display DataFrame contents |
| __`take(n)`__ | Return the first n rows from a DataFrame |
| __`first()`__ | Return the first row from a DataFrame |
| __`write`__ | Save a DataFrame to storage |

For more information see the [DataFrame API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html).

## D. DataFrame API Optimizations

<p style="font-size: 1em; line-height: 1.6; color: #333">Because DataFrame operations are declarative, Spark is free to optimize how they execute:</p>

- **Adaptive Query Execution (AQE)** - dynamic plan adjustments (cost-based optimization) during runtime based on actual data characteristics and execution patterns
- **In-Memory Columnar Storage** - an in-memory columnar format for DataFrames enabling efficient analytical query performance and a reduced memory footprint
- **Built-in Statistics** - automatic statistics collection when saving to optimized formats (Parquet, Delta) enables smarter query planning and execution
- **Catalyst Optimizer and Photon** - discussed below

## E. DataFrame (or Query) Planning

<p style="font-size: 1em; line-height: 1.6; color: #333">When a DataFrame is evaluated, the Driver creates an optimized execution plan through a series of transformations - converting the logical plan into an efficient physical execution plan that minimizes resource usage and execution time.</p>

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">From unresolved plan to executed code</strong>
      </div>
    </div>
  </summary>
  <div style="margin: 8px 0 4px 0; overflow-x: auto">
    <svg viewBox="0 0 1100 175" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Catalyst planning stages from unresolved logical plan to execution" style="width:100%; max-width:1100px; height:auto; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif">
      <defs>
        <marker id="plan-arrow" markerWidth="10" markerHeight="8" refX="7" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L8,3 L0,6 Z" fill="#618794"/></marker>
      </defs>
      <rect x="15" y="80" width="150" height="60" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="90" y="106" text-anchor="middle" font-size="14" font-weight="600" fill="#333">Unresolved</text>
      <text x="90" y="125" text-anchor="middle" font-size="14" fill="#333">Logical Plan</text>
      <rect x="245" y="80" width="150" height="60" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="320" y="106" text-anchor="middle" font-size="14" font-weight="600" fill="#333">Analyzed</text>
      <text x="320" y="125" text-anchor="middle" font-size="14" fill="#333">Logical Plan</text>
      <rect x="475" y="80" width="150" height="60" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="550" y="106" text-anchor="middle" font-size="14" font-weight="600" fill="#333">Optimized</text>
      <text x="550" y="125" text-anchor="middle" font-size="14" fill="#333">Logical Plan</text>
      <rect x="705" y="80" width="150" height="60" rx="8" fill="#FFFFFF" stroke="#DCE0E2" stroke-width="1.5"/>
      <text x="780" y="106" text-anchor="middle" font-size="14" font-weight="600" fill="#333">Physical</text>
      <text x="780" y="125" text-anchor="middle" font-size="14" fill="#333">Plan</text>
      <rect x="935" y="80" width="150" height="60" rx="8" fill="#FFCC66" stroke="#CD7F32" stroke-width="2"/>
      <text x="1010" y="106" text-anchor="middle" font-size="14" font-weight="700" fill="#333">Execution on</text>
      <text x="1010" y="125" text-anchor="middle" font-size="14" fill="#333">the Cluster</text>
      <line x1="167" y1="110" x2="243" y2="110" stroke="#618794" stroke-width="2" marker-end="url(#plan-arrow)"/>
      <line x1="397" y1="110" x2="473" y2="110" stroke="#618794" stroke-width="2" marker-end="url(#plan-arrow)"/>
      <line x1="627" y1="110" x2="703" y2="110" stroke="#618794" stroke-width="2" marker-end="url(#plan-arrow)"/>
      <line x1="857" y1="110" x2="933" y2="110" stroke="#618794" stroke-width="2" marker-end="url(#plan-arrow)"/>
      <text x="205" y="97" text-anchor="middle" font-size="12" fill="#1B5162">Analysis</text>
      <text x="435" y="90" text-anchor="middle" font-size="12" fill="#1B5162">Logical</text>
      <text x="435" y="103" text-anchor="middle" font-size="12" fill="#1B5162">Optimization</text>
      <text x="665" y="90" text-anchor="middle" font-size="12" fill="#1B5162">Physical</text>
      <text x="665" y="103" text-anchor="middle" font-size="12" fill="#1B5162">Planning</text>
      <text x="895" y="90" text-anchor="middle" font-size="12" fill="#1B5162">Code</text>
      <text x="895" y="103" text-anchor="middle" font-size="12" fill="#1B5162">Generation</text>
    </svg>
  </div>
</details>
<br/>
<details>
<summary style="cursor: pointer; list-style: none; user-select: none">
<div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: center; gap: 12px">
<span>&#x25B6;</span>
<strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
</div>
</div>
</summary>
<div style="border-left: 4px solid #1B5162; padding: 4px 24px 14px 24px; margin: -16px 0 16px 0">
<div style="font-size: 1.2em; font-weight: bold; color: #1B5162; margin: 12px 0 4px 0">The Planning Phases</div>
<ul style="font-size: 1.1em; line-height: 1.45; color: #333; margin: 0 0 10px 0; padding-left: 24px">
<li style="margin: 4px 0"><strong>Analysis</strong> resolves names against the catalog: which table does <code>customers</code> refer to, what type is <code>account_balance</code>, does the referenced column exist?</li>
<li style="margin: 4px 0"><strong>Logical optimization</strong> applies rule-based rewrites such as predicate pushdown (filter as early as possible), column pruning (read only the columns used), and constant folding.</li>
<li style="margin: 4px 0"><strong>Physical planning</strong> selects concrete execution strategies - for example choosing a broadcast join over a shuffle join when one side is small.</li>
<li style="margin: 4px 0"><strong>Code generation</strong> compiles the chosen plan into efficient JVM bytecode (or hands it to Photon for vectorized native execution on Databricks).</li>
<li style="margin: 4px 0">You can inspect the plans for any DataFrame with <code>df.explain()</code> - the ETL demo in Section 4 does exactly this.</li>
</ul>
</div>
</details>

### Catalyst Optimizer and Photon

**Catalyst Optimizer:**

- The query optimization engine that converts DataFrame operations into an optimized execution plan
- Applies rule-based and cost-based optimizations to improve query performance

**Photon Engine:**

- The Databricks-native vectorized query engine that accelerates query execution
- Processes data in batches rather than row-by-row for better performance
- Runs by default on Databricks SQL warehouses and serverless compute
- Can be enabled on All-Purpose and Job Clusters in Databricks

## F. More on Columnar Storage

<p style="font-size: 1em; line-height: 1.6; color: #333">Columnar storage organizes data by column rather than by row, enabling efficient scanning and analysis. It is particularly efficient for analytical workloads and is implemented both in the DataFrame internal storage format and in physical file formats such as <strong>Parquet</strong> and ORC.</p>

<p style="font-size: 1em; line-height: 1.6; color: #333">In <strong>row-oriented storage</strong>, all the values of a record are stored together. Reading one column means reading every row in full:</p>

<table style="border-collapse: collapse; margin: 8px 0">
<tr><td style="border: 1px solid #999; padding: 6px 14px; background: #FABFBA">id: 1</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FABFBA">name: Ada</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FABFBA">amt: 100</td></tr>
<tr><td style="border: 1px solid #999; padding: 6px 14px; background: #DCE0E2">id: 2</td><td style="border: 1px solid #999; padding: 6px 14px; background: #DCE0E2">name: Ben</td><td style="border: 1px solid #999; padding: 6px 14px; background: #DCE0E2">amt: 250</td></tr>
<tr><td style="border: 1px solid #999; padding: 6px 14px; background: #FFE6B3">id: 3</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FFE6B3">name: Cy</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FFE6B3">amt: 75</td></tr>
</table>

<p style="font-size: 1em; line-height: 1.6; color: #333">In <strong>column-oriented storage</strong>, all the values of a column are stored together. A query that sums <code>amt</code> touches only the last block, skipping the rest entirely:</p>

<table style="border-collapse: collapse; margin: 8px 0">
<tr><td style="border: 1px solid #999; padding: 6px 14px; background: #FABFBA">id: 1</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FABFBA">id: 2</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FABFBA">id: 3</td></tr>
<tr><td style="border: 1px solid #999; padding: 6px 14px; background: #DCE0E2">name: Ada</td><td style="border: 1px solid #999; padding: 6px 14px; background: #DCE0E2">name: Ben</td><td style="border: 1px solid #999; padding: 6px 14px; background: #DCE0E2">name: Cy</td></tr>
<tr><td style="border: 1px solid #999; padding: 6px 14px; background: #FFE6B3">amt: 100</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FFE6B3">amt: 250</td><td style="border: 1px solid #999; padding: 6px 14px; background: #FFE6B3">amt: 75</td></tr>
</table>

<p style="font-size: 1em; line-height: 1.6; color: #333">Columnar layouts also compress better, because values within a column are of the same type and often similar.</p>

## Summary

- DataFrames are distributed, immutable collections of records with a schema
- Transformations describe work lazily; actions trigger execution of the accumulated plan
- The Catalyst optimizer turns your code into an optimized physical plan; Photon accelerates execution on Databricks
- Columnar storage (in memory and in formats like Parquet) makes analytical scans and aggregations efficient

## Knowledge Check

Answer the questions below to check your understanding of this lecture.

<div id="mcqRoot"></div>

<script>
(function() {
    var questions = [
        {
                "question": "DataFrames are evaluated using ____ evaluation.",
                "options": [
                        {
                                "id": "a",
                                "text": "immediate"
                        },
                        {
                                "id": "b",
                                "text": "real time"
                        },
                        {
                                "id": "c",
                                "text": "delayed"
                        },
                        {
                                "id": "d",
                                "text": "lazy"
                        }
                ],
                "correct": "d",
                "explanation": "Transformations only build up a logical plan. Execution is deferred until an action (such as count(), show(), or write) is requested - this is lazy evaluation."
        },
        {
                "question": "Which DataFrame API transformation is the functional equivalent of a WHERE clause in a SQL statement?",
                "options": [
                        {
                                "id": "a",
                                "text": "select()"
                        },
                        {
                                "id": "b",
                                "text": "filter()"
                        },
                        {
                                "id": "c",
                                "text": "distinct()"
                        },
                        {
                                "id": "d",
                                "text": "like()"
                        }
                ],
                "correct": "b",
                "explanation": "filter() (and its alias where()) restricts rows based on a condition, exactly like a SQL WHERE clause. select() corresponds to the SELECT column list."
        },
        {
                "question": "Which statement about DataFrame operations in Spark is correct?",
                "options": [
                        {
                                "id": "a",
                                "text": "They modify the existing DataFrame directly"
                        },
                        {
                                "id": "b",
                                "text": "They are executed immediately upon definition"
                        },
                        {
                                "id": "c",
                                "text": "They are distributed automatically across partitions"
                        },
                        {
                                "id": "d",
                                "text": "They cannot be mapped to SQL operations"
                        }
                ],
                "correct": "c",
                "explanation": "DataFrames are immutable and lazily evaluated, and their operations map directly to SQL equivalents. What is true is that operations are automatically distributed across the partitions of the DataFrame."
        }
];
    
    var currentIndex = 0;
    var score = 0;
    var selectedAnswer = null;
    var showingFeedback = false;
    
    function init() {
        currentIndex = 0;
        score = 0;
        selectedAnswer = null;
        showingFeedback = false;
        render();
    }
    
    function render() {
        var root = document.getElementById('mcqRoot');
        var html = '';
        var isComplete = currentIndex >= questions.length;
        
        html += '<div style="font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; width: 100%; margin: 10px 0; padding: 24px; background: #f5f7fa; border-radius: 12px; border: 1px solid #e0e0e0; box-sizing: border-box;">';
        
        // Header
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div>';
        html += '<div style="font-size: 1.4em; font-weight: 600; color: #333;">📝 Multiple Choice Quiz</div>';
        html += '<div style="font-size: 0.95em; color: #666; margin-top: 4px;">Knowledge Check</div>';
        html += '</div>';
        html += '<div style="display: flex; gap: 12px; align-items: center;">';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e8f5e9; color: #2e7d32;">Score: ' + score + '/' + questions.length + '</span>';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e3f2fd; color: #1976d2;">Q' + (currentIndex + 1) + '/' + questions.length + '</span>';
        html += '</div></div>';
        
        if (isComplete) {
            // Results
            var percentage = Math.round(score / questions.length * 100);
            html += '<div style="text-align: center; padding: 40px 20px; background: #fff; border-radius: 12px; border: 2px solid #e0e0e0;">';
            html += '<div style="font-size: 3em; margin-bottom: 16px;">' + (percentage >= 75 ? '🏆' : percentage >= 50 ? '👍' : '📚') + '</div>';
            html += '<div style="font-size: 1.8em; font-weight: 700; color: #333; margin-bottom: 8px;">Quiz Complete!</div>';
            html += '<div style="font-size: 1.2em; color: #666; margin-bottom: 24px;">You scored ' + score + ' out of ' + questions.length + ' (' + percentage + '%)</div>';
            html += '<button id="retakeBtn" style="padding: 14px 32px; background: #1976d2; color: white; border: none; border-radius: 8px; cursor: pointer; font-size: 1.1em; font-weight: 600;">Retake Quiz</button>';
            html += '</div>';
        } else {
            var q = questions[currentIndex];
            
            // Question
            html += '<div style="background: #fff; border-radius: 12px; border: 2px solid #e0e0e0; padding: 24px; margin-bottom: 20px;">';
            html += '<div style="font-size: 1.15em; color: #333; line-height: 1.6;">' + q.question + '</div>';
            html += '</div>';
            
            // Options
            html += '<div style="display: flex; flex-direction: column; gap: 12px; margin-bottom: 20px;">';
            
            q.options.forEach(function(opt) {
                var isSelected = selectedAnswer === opt.id;
                var isCorrect = opt.id === q.correct;
                var bgColor = '#fff';
                var borderColor = '#e0e0e0';
                var textColor = '#333';
                
                if (showingFeedback) {
                    if (isCorrect) {
                        bgColor = '#e8f5e9';
                        borderColor = '#4caf50';
                        textColor = '#2e7d32';
                    } else if (isSelected && !isCorrect) {
                        bgColor = '#ffebee';
                        borderColor = '#f44336';
                        textColor = '#c62828';
                    }
                } else if (isSelected) {
                    bgColor = '#e3f2fd';
                    borderColor = '#1976d2';
                    textColor = '#1565c0';
                }
                
                var cursor = showingFeedback ? 'default' : 'pointer';
                
                html += '<div class="mcq-option" data-id="' + opt.id + '" style="display: flex; align-items: center; gap: 16px; padding: 16px 20px; background: ' + bgColor + '; border: 2px solid ' + borderColor + '; border-radius: 8px; cursor: ' + cursor + '; transition: all 0.15s ease;">';
                html += '<div style="width: 32px; height: 32px; border: 2px solid ' + borderColor + '; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: 700; color: ' + textColor + '; background: ' + (isSelected || (showingFeedback && isCorrect) ? borderColor : 'transparent') + '; color: ' + (isSelected || (showingFeedback && isCorrect) ? '#fff' : textColor) + ';">' + opt.id.toUpperCase() + '</div>';
                html += '<div style="flex: 1; font-size: 1em; color: ' + textColor + ';">' + opt.text + '</div>';
                if (showingFeedback && isCorrect) {
                    html += '<span style="color: #4caf50; font-size: 1.2em;">✓</span>';
                } else if (showingFeedback && isSelected && !isCorrect) {
                    html += '<span style="color: #f44336; font-size: 1.2em;">✗</span>';
                }
                html += '</div>';
            });
            
            html += '</div>';
            
            // Feedback
            if (showingFeedback) {
                var isCorrect = selectedAnswer === q.correct;
                html += '<div style="background: ' + (isCorrect ? '#e8f5e9' : '#fff3e0') + '; border: 2px solid ' + (isCorrect ? '#4caf50' : '#ff9800') + '; border-radius: 8px; padding: 16px 20px; margin-bottom: 20px;">';
                html += '<div style="font-weight: 600; color: ' + (isCorrect ? '#2e7d32' : '#e65100') + '; margin-bottom: 8px;">' + (isCorrect ? '✓ Correct!' : '✗ Not quite') + '</div>';
                html += '<div style="color: #333; font-size: 0.95em; line-height: 1.5;">' + q.explanation + '</div>';
                html += '</div>';
            }
            
            // Buttons
            html += '<div style="display: flex; gap: 12px; justify-content: flex-end;">';
            if (!showingFeedback) {
                html += '<button id="submitBtn" style="padding: 12px 28px; background: ' + (selectedAnswer ? '#4caf50' : '#ccc') + '; color: white; border: none; border-radius: 6px; cursor: ' + (selectedAnswer ? 'pointer' : 'not-allowed') + '; font-size: 1em; font-weight: 600;"' + (selectedAnswer ? '' : ' disabled') + '>Submit Answer</button>';
            } else {
                html += '<button id="nextQBtn" style="padding: 12px 28px; background: #1976d2; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 1em; font-weight: 600;">' + (currentIndex < questions.length - 1 ? 'Next Question' : 'See Results') + '</button>';
            }
            html += '</div>';
        }
        
        html += '</div>';
        
        root.innerHTML = html;
        attachEvents();
    }
    
    function attachEvents() {
        var options = document.querySelectorAll('.mcq-option');
        var submitBtn = document.getElementById('submitBtn');
        var nextQBtn = document.getElementById('nextQBtn');
        var retakeBtn = document.getElementById('retakeBtn');
        
        options.forEach(function(opt) {
            opt.addEventListener('click', function() {
                if (!showingFeedback) {
                    selectedAnswer = this.dataset.id;
                    render();
                }
            });
            
            if (!showingFeedback) {
                opt.addEventListener('mouseenter', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#1976d2';
                        this.style.background = '#f5f9ff';
                    }
                });
                opt.addEventListener('mouseleave', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#e0e0e0';
                        this.style.background = '#fff';
                    }
                });
            }
        });
        
        if (submitBtn) {
            submitBtn.addEventListener('click', function() {
                if (selectedAnswer) {
                    var q = questions[currentIndex];
                    if (selectedAnswer === q.correct) {
                        score++;
                    }
                    showingFeedback = true;
                    render();
                }
            });
        }
        
        if (nextQBtn) {
            nextQBtn.addEventListener('click', function() {
                currentIndex++;
                selectedAnswer = null;
                showingFeedback = false;
                render();
            });
        }
        
        if (retakeBtn) {
            retakeBtn.addEventListener('click', function() {
                init();
            });
        }
    }
    
    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }
    
    init();
})();
</script>

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>